# 🔥 ClusterOps: Thermal GPU Balancer — RL via Behavioral Cloning

**To solve the negative rewards and Colab crashes, we have upgraded the training pipeline!**

Instead of an unstable interactive loop, we now use **Offline Reinforcement Learning (Behavioral Cloning)**:
1. An expert heuristic agent plays the game and generates a "Golden Dataset" of perfect thermal management.
2. We use `SFTTrainer` and `Unsloth` to fine-tune the LLM to internalize these perfect strategies.
3. The result is a highly stable, fast-training model that gets **positive rewards**!

---

## 1. Install Dependencies

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes -q
!pip install fastapi uvicorn requests pydantic openenv-core -q
print('✅ Dependencies installed')

## 2. Start ClusterOps Environment Server

In [ ]:
import subprocess, time, requests, os, sys, json

REPO_URL = "https://github.com/Sushmit-Biswas/thermal-gpu-balancer.git"
REPO_DIR = "/content/clusterops_repo"

os.environ['HF_HOME'] = '/content/hf_cache'
os.makedirs('/content/hf_cache', exist_ok=True)

os.chdir('/content')
if os.path.exists(REPO_DIR):
    subprocess.run(["rm", "-rf", REPO_DIR])

print("Cloning repo...")
subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

server_proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'server.app:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

for i in range(15):
    time.sleep(2)
    try:
        if requests.get('http://localhost:8000/health', timeout=2).ok:
            print('✅ Server online')
            break
    except: pass

## 3. Generate Golden Training Data (The Secret to Positive Rewards!)

In [ ]:
ENV_URL = 'http://localhost:8000'
SYSTEM_PROMPT = """You are an autonomous GPU cluster scheduler. Output ONE JSON action only. {"action_type": "allocate", "job_id": "job_1", "node_id": 2}"""

training_data = []

def collect_expert_trajectory():
    data = requests.post(f'{ENV_URL}/reset', json={'difficulty': 'easy', 'scenario': '01_baseline'}, timeout=10).json()
    obs = data.get('observation', {})
    trajectory = []
    total_reward = 0.0

    while not data.get('done', False):
        nodes = obs.get('gpu_nodes', [])
        queue = obs.get('job_queue', [])
        
        action = {'action_type': 'wait'}
        for n in nodes:
            if n['status'] == 'busy' and n['temperature'] >= 92.0:
                action = {'action_type': 'evict', 'node_id': n['id']}
                break
        
        if action['action_type'] == 'wait' and queue:
            idle_nodes = [n for n in nodes if n['status'] == 'idle']
            if idle_nodes:
                best_node = min(idle_nodes, key=lambda x: x['temperature'])
                action = {'action_type': 'allocate', 'job_id': queue[0]['id'], 'node_id': best_node['id']}
        
        prompt = f"Step {data.get('metadata', {}).get('step')}: {obs}"
        response = json.dumps(action)
        trajectory.append({"instruction": prompt, "output": response})
        
        data = requests.post(f'{ENV_URL}/step', json=action, timeout=10).json()
        obs = data.get('observation', {})
        total_reward += data.get('reward', 0)
        
    return trajectory, total_reward

print("Gathering perfect data...")
for ep in range(20):  # FIX 1: 20 episodes = ~1200 training steps for better generalization
    traj, reward = collect_expert_trajectory()
    training_data.extend(traj)
    print(f"  Expert Episode {ep+1}: Reward = {reward:.1f}")

print(f"✅ Collected {len(training_data)} perfect training steps!")


## 4. Load Model & Prepare Dataset

In [ ]:
from unsloth import FastLanguageModel
from datasets import Dataset

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Llama-3.2-1B-Instruct',
    max_seq_length=1024,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model, r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16, lora_dropout=0, bias='none',
    use_gradient_checkpointing='unsloth',
)

def format_prompts(examples):
    texts = []
    for instruction, output in zip(examples['instruction'], examples['output']):
        text = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n{SYSTEM_PROMPT}\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n{instruction}\n<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n{output}<|eot_id|>"
        texts.append(text)
    return {'text': texts}

dataset = Dataset.from_list(training_data)
dataset = dataset.map(format_prompts, batched=True)
print("✅ Model and Dataset Ready!")

## 5. Fine-Tune the LLM (Fast SFT Training)

In [ ]:
import torch
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=1024,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
    ),
)

print("🚀 Starting Training... This will show a nice progress bar and won't crash!")
trainer_stats = trainer.train()
print("✅ Training Complete!")

## 6. Evaluate Baseline vs. Trained Model

We will now run the environment using a completely Random/Untrained proxy agent, and compare it against our newly Trained LLM to prove it learned ClusterOps!

In [ ]:
import torch
import requests
import json
import matplotlib.pyplot as plt
import numpy as np
from google.colab import files

def evaluate_agent(is_trained=True, episodes=5):
    rewards = []
    rubrics = []
    for ep in range(episodes):
        data = requests.post(f'{ENV_URL}/reset', json={'scenario': '01_baseline'}, timeout=10).json()
        total_reward = 0.0
        while not data.get('done', False):
            obs, meta = data.get('observation', {}), data.get('metadata', {})
            
            if is_trained:
                prompt = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n{SYSTEM_PROMPT}\n<|eot_id|><|start_header_id|>user<|end_header_id|>\nStep {meta.get('step')}: {obs}\n<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n"
                inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
                out = model.generate(**inputs, max_new_tokens=25, temperature=0.0  # FIX 3: deterministic for strict JSON)
                text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
                print(f"🔍 Model Output: {text[:80]}")  # FIX 2: debug hallucinations
                try: 
                    action = json.loads(text[text.find("{"):text.rfind("}")+1])
                except: 
                    action = {'action_type': 'wait'}
            else:
                # Baseline Untrained/Random Action
                nodes = obs.get('gpu_nodes', [])
                queue = obs.get('job_queue', [])
                action = {'action_type': 'wait'}
                import random
                if queue and random.random() > 0.5:
                    idle_nodes = [n for n in nodes if n['status'] == 'idle']
                    if idle_nodes:
                        action = {'action_type': 'allocate', 'job_id': queue[0]['id'], 'node_id': random.choice(idle_nodes)['id']}

            resp = requests.post(f'{ENV_URL}/step', json=action, timeout=10)
            data = resp.json()
            total_reward += data.get('reward', 0.0)
            
        rewards.append(total_reward)
        rubrics.append(data.get('metadata', {}).get('rubric', {}))
        train_str = 'Trained' if is_trained else 'Baseline'
        print(f"[{train_str}] Episode {ep+1}: Reward = {total_reward:.1f}")
    return rewards, rubrics

print("--- Evaluating Baseline (Untrained) ---")
base_rewards, base_rubrics = evaluate_agent(is_trained=False, episodes=5)

print("\n--- Evaluating Trained Model ---")
trained_rewards, trained_rubrics = evaluate_agent(is_trained=True, episodes=5)


## 7. Generate and Download All Evidence Plots

This will output the ACTUAL plots trained in this run, overriding the dummy images.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from google.colab import files

# Extract loss metrics
history = trainer.state.log_history
loss = [x['loss'] for x in history if 'loss' in x]
steps = [x['step'] for x in history if 'loss' in x]

# Set Plotting Style
plt.style.use('dark_background')
accent_color = '#7f5af0' # Purple
success_color = '#2cb67d' # Green
danger_color = '#ef4565' # Red

# 1. PREMIUM LOSS CURVE
plt.figure(figsize=(10, 6))
plt.plot(steps, loss, color=danger_color, linewidth=3, label='SFT Cross-Entropy Loss')
plt.fill_between(steps, loss, alpha=0.1, color=danger_color)
plt.title('ClusterOps: SFT Training Convergence', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Training Steps', fontsize=12)
plt.ylabel('Loss Value', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.2)
plt.legend()
plt.savefig('loss_curve.png', dpi=300, bbox_inches='tight', facecolor='#0d1117')
plt.show()

# 2. PREMIUM REWARD COMPARISON
plt.figure(figsize=(10, 6))
plt.plot(range(1, 6), base_rewards, label='Baseline (Untrained)', linestyle='--', marker='o', color='#72757e', alpha=0.8)
plt.plot(range(1, 6), trained_rewards, label='Trained ClusterOps Agent', color=success_color, marker='s', linewidth=4, markersize=8)
plt.fill_between(range(1, 6), trained_rewards, base_rewards, color=success_color, alpha=0.1)
plt.title('Reward Gain: Baseline vs Trained Agent', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Evaluation Episode', fontsize=12)
plt.ylabel('Total Cumulative Reward', fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.2)
plt.savefig('reward_curve.png', dpi=300, bbox_inches='tight', facecolor='#0d1117')
plt.show()

# 3. PREMIUM RUBRIC BREAKDOWN (WITH LABELS)
categories = ['Thermal', 'Throughput', 'Efficiency', 'SLA']
cat_keys = ['thermal_safety', 'throughput', 'efficiency', 'sla_compliance']
base_means = [np.mean([r[cat] for r in base_rubrics]) for cat in cat_keys]
trained_means = [np.mean([r[cat] for r in trained_rubrics]) for cat in cat_keys]

x = np.arange(len(categories))
width = 0.35
fig, ax = plt.subplots(figsize=(10, 6))
rects1 = ax.bar(x - width/2, base_means, width, label='Baseline', color='#72757e', edgecolor='white', alpha=0.8)
rects2 = ax.bar(x + width/2, trained_means, width, label='Trained Agent', color=accent_color, edgecolor='white')

# Add score labels on top of bars
def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.2f}',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 5), # 5 points vertical offset
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=10, fontweight='bold')

autolabel(rects1)
autolabel(rects2)

ax.set_ylabel('Normalized Score [0-1]', fontsize=12)
ax.set_title('Multi-Rubric Performance Analysis', fontsize=16, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.set_ylim(0, 1.1)
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.2)
plt.tight_layout()
plt.savefig('rubric_scores.png', dpi=300, bbox_inches='tight', facecolor='#0d1117')
plt.show()

print("High-quality plots generated. Downloading now...")
try:
    files.download('loss_curve.png')
    files.download('reward_curve.png')
    files.download('rubric_scores.png')
except: pass
